# SHAP Analysis — LightGBM

This notebook uses **SHAP (SHapley Additive exPlanations)** to explain the predictions of the LightGBM model, both globally (which features drive risk overall) and locally (why a specific customer received a given PD).

### Methodological note

The production model is **calibrated**: the raw LightGBM score is passed through a calibration layer to produce reliable PDs. SHAP's `TreeExplainer`, which is exact and fast, only works with the tree model itself. We therefore explain the **underlying LightGBM model**, whose output is in **log-odds**.

Since calibration is a monotonically increasing transformation, feature rankings and effect directions are the same for the calibrated model. Only the scale differs: SHAP values here are in log-odds, not in probability points.

**How to read SHAP values:** each customer's log-odds is the sum of a base value (the average model output) and one SHAP value per feature. Positive values **increase** the predicted risk; negative values **decrease** it.

In [8]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import joblib
import shap
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

from src.config import X_VAL_DIR, Y_VAL_DIR, MODELS, MODELS_CALIBRATED
from src.processing import process_data, to_feature_frame

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Parameters

- `MODEL_NAME`: key of the model in `MODELS_CALIBRATED`.
- `SAMPLE_SIZE`: number of customers used in the analysis. `TreeExplainer` is fast, so a few thousand is fine; set it to `None` to use the full validation set.
- `THRESHOLD`: PD cut-off used to pick example customers for local explanations (approved vs. rejected).
- `TOP_N`: number of top features shown in the dependence plots.

In [9]:
MODEL_NAME = "lightgbm"
SAMPLE_SIZE = 5000
THRESHOLD = 0.10
TOP_N = 4
RANDOM_STATE = 42

## 2. Load the data and the models

The calibrated model wraps the trained LightGBM in a `FrozenEstimator` and applies a **sigmoid (Platt) calibration** on top of its output. Therefore:

- the **raw LightGBM model** (`MODELS`) is explained with SHAP;
- the **calibrated model** (`MODELS_CALIBRATED`) is used only to obtain the final PDs for the local examples.

Both models use the same processed features.

In [10]:
X_val = pd.read_csv(X_VAL_DIR, index_col="ID")
y_val = pd.read_csv(Y_VAL_DIR, index_col="ID").squeeze("columns")

raw_config = MODELS[MODEL_NAME]
calibrated_config = MODELS_CALIBRATED[MODEL_NAME]

X_val_processed = process_data(model=raw_config["type"], X=X_val)

raw_model = joblib.load(raw_config["path"])
calibrated_model = joblib.load(calibrated_config["path"])

print("Raw model:", type(raw_model).__name__)
print("Processed data:", X_val_processed.shape)

Raw model: LGBMClassifier
Processed data: (22500, 14)


### Check: calibration preserves the ranking

SHAP explains the raw model, so we confirm that the calibrated PD is a monotonic transformation of the raw score. A Spearman correlation of 1.0 means both models rank customers in exactly the same order. In that case, feature importance and effect directions found with SHAP also hold for the calibrated model.

In [11]:
raw_pd = raw_model.predict_proba(X_val_processed)[:, 1]
calibrated_pd = calibrated_model.predict_proba(X_val_processed)[:, 1]

rho, _ = spearmanr(raw_pd, calibrated_pd)
print(f"Spearman correlation (raw vs. calibrated): {rho:.6f}")

Spearman correlation (raw vs. calibrated): 1.000000


## 3. Compute SHAP values

We draw a random sample of customers and compute SHAP values with `TreeExplainer`. For a binary LightGBM model, some SHAP versions return values for both classes; in that case we keep class `1` (default).

In [18]:
X_val_processed = to_feature_frame(X_val_processed, index=X_val.index)

print(X_val_processed.shape)
print(X_val_processed.index.equals(X_val.index))

X_val_processed.head(10)

(22500, 14)
True


,Util,Age,Late30_59,Debt,Income,OpenCred,Late90,RealEstate,Late60_89,AnyLate,TotalLate,LateScore,UtilAbove100,No.Dep
ID,,,,,,,,,,,,,,
24378,0.400263,49.0,0.0,0.349646,8050.0,14.0,0.0,1.0,0.0,0.0,0.0,0.0,3.0,0.0
77975,1.000000,37.0,0.0,0.000000,5382.0,0.0,1.0,0.0,0.0,1.0,3.0,0.0,0.0,1.0
23942,0.727199,81.0,5.0,0.273762,5975.0,7.0,0.0,1.0,1.0,6.0,7.0,0.0,0.0,1.0
6623,0.047398,60.0,0.0,0.443198,11055.0,6.0,0.0,4.0,0.0,0.0,0.0,0.0,2.0,0.0
78152,0.000690,86.0,0.0,0.000445,2246.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
133303,0.115854,59.0,0.0,0.563170,5500.0,19.0,0.0,2.0,0.0,0.0,0.0,0.0,2.0,0.0
52632,0.063847,50.0,0.0,0.003123,12166.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0
119522,0.501007,34.0,1.0,0.099956,2260.0,4.0,0.0,0.0,1.0,2.0,3.0,0.0,0.0,1.0
71182,0.181088,54.0,0.0,0.288341,6200.0,20.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0
